# TalentDesk, Section 4 Lab (Exercise): Hooks for Gating, Normalization, and Compliance

A hands-on exercise built on the **Claude Agent SDK**, running **Sonnet** (`claude-sonnet-4-6`).
It combines the two Section 4 skills: moving a guarantee out of the prompt and into **code** with a
`PreToolUse` gate and a `PostToolUse` recorder (Lab 1), and using hooks to **normalize** raw tool
output and **enforce a compliance threshold** (Lab 2). You fill in four short `TODO` blocks;
everything else is provided. Every hook is testable offline by calling it with a mock payload, so no
key or Node.js is needed to check your work, and a full solution is at the end.

## The real-world scenario

TalentDesk's recruiter agent can now **send offers**, an action that commits real money and a
real promise to a candidate. "Always run the background check before sending an offer" and "never
send an offer above the salary band without sign-off" are fine sentences to put in a prompt, and a
model will usually follow them. Usually is not good enough when an offer letter is on the line. The
same rules written as **hooks** cannot be talked out of: the offer tool simply does not run until
its prerequisite is met and its amount is within the band.

There is a second problem: the ATS returns raw data, a Unix timestamp for last activity and a
numeric stage code, and if the model reasons over that raw shape it makes small, avoidable mistakes.
That is fixed at the **hook** layer too, where you can reshape output before the model ever sees it.

The question this lab answers: **how do you make a workflow prerequisite and a compliance limit
guarantees rather than suggestions, and how do you normalize tool output before the model uses
it?**

## Objectives

- Enforce a **prerequisite** with a `PreToolUse` gate: no `send_offer` until `background_check` has
  cleared the candidate. Record the clearance with a `PostToolUse` hook.
- **Normalize** a raw ATS payload (Unix time to ISO, stage code to a word) so every later step
  reasons over clean data.
- Add a **compliance rule**: deny an offer over the salary band so a human can approve it.
- See the difference between **prompt-based** enforcement (best effort) and **programmatic**
  enforcement (deterministic), and hand off a case in a **structured** shape.

## The outcome you should reach

By the end you will have:

- a `PreToolUse` gate that blocks `send_offer` until the candidate is cleared, and a `PostToolUse`
  recorder that unlocks it;
- a `normalize()` transform and the hook that appends its clean view after a raw tool result;
- a `check_band()` compliance rule and the gate that denies an over-band offer while letting a
  within-band one through;
- and a structured escalation packet that validates every time.

Target time: **20 to 30 minutes.** Four small `TODO` blocks. Every hook and rule is testable offline
by awaiting it with a mock payload; the live agent needs a real key and Node.js 18+.

## How to run

Run top to bottom. The pure-Python rules and the hooks are all testable offline: the self-checks
call each hook with a mock payload and assert its decision. To run the real agent, paste a real key
into **Setup 2/3**, re-run from the top, and have Node.js 18+ installed. Prevention lives in
`PreToolUse`; `PostToolUse` shapes and records but cannot undo.

## 0. Setup

**This cell:** installs the packages. The **Agent SDK** provides the tools and hooks API; the
base SDK and dotenv handle the key. The Agent SDK also needs Node.js 18+, which cannot be
pip-installed; the offline hook self-checks do not need it.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` so async hooks can
be called like ordinary functions (the offline self-checks use it too).

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # read the API key from the environment
import sys                                      # detect Windows (it needs a special event loop)
import json                                     # build tool payloads and escalation packets
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)
from datetime import datetime, timezone         # for the timestamp conversion in normalize()

try:                                            # load a .env file if present
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the agent will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any coroutine (a hook or a live call), notebook-safe
    box = {}
    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:    box["value"] = loop.run_until_complete(make_coro())
        except Exception as e: box["error"] = e
        finally: loop.close()
    t = threading.Thread(target=worker); t.start(); t.join()
    if "error" in box: raise box["error"]
    return box.get("value")

print("live model calls:", "ON" if RUN_LIVE else "OFF (hooks tested offline with mock payloads)")

**This cell:** the shared **TalentDesk world** and the **rails** the hooks use. `CLEARED` is
the set of candidates whose background check passed (the prerequisite rail); `BAND_LIMIT` is the
salary ceiling the compliance gate enforces.

In [ ]:
# ===== SETUP 3/3 - the shared data and the rails =====
CANDIDATES = {"C1": {"name": "Ana", "stage": 3},   # a tiny candidate book (stage code for normalize)
              "C2": {"name": "Ben", "stage": 2}}
STAGE_NAMES = {1: "applied", 2: "screening", 3: "interview", 4: "offer"}   # code -> word

CLEARED = set()                                    # candidates cleared by background_check (the rail)
BAND_LIMIT = 180000.0                              # offers over this need human approval (compliance)

def send_offer(candidate_id, salary):              # the plain offer action (used in the contrast below)
    return f"offer sent to {candidate_id} at {salary}"   # the gate, not the body, is the point

print("candidates:", list(CANDIDATES), "| cleared starts empty | band limit:", BAND_LIMIT)

### Prompt-only versus programmatic enforcement

A rule in a **prompt** is advisory: the model reads it and usually complies, but nothing stops it
from skipping the step. A rule in **code** (a hook) is a control point in the execution lifecycle: it
runs before the tool and can **deny** it outright, so the prerequisite holds every time regardless of
wording. Offer-touching steps belong in code.

In [ ]:
# ===== prompt-only vs programmatic, in plain Python (provided) =====
def unguarded_offer(candidate_id, salary, cleared):   # prompt-only world: the rule lives only in text
    return send_offer(candidate_id, salary)           #   nothing in CODE stops an uncleared offer

def guarded_offer(candidate_id, salary, cleared):     # programmatic world: the rule lives in code
    if candidate_id not in cleared:                    #   the prerequisite, checked deterministically
        return "BLOCKED: run the background check first"
    return send_offer(candidate_id, salary)

demo = set()
print("unguarded, uncleared:", unguarded_offer("C1", 150000, demo))   # sends anyway (bad)
print("guarded,   uncleared:", guarded_offer("C1", 150000, demo))     # blocked (good)
demo = {"C1"}
print("guarded,   cleared:  ", guarded_offer("C1", 150000, demo))     # allowed

---

### 🎯 Part A - make the prerequisite a guarantee

Hooks receive a dict, `input_data`, whose `tool_name` is the tool about to run (or that just ran) and
whose `tool_input` holds its arguments. A `PreToolUse` hook returns a **deny** decision to block a
tool, or an empty dict `{}` to allow it. A `PostToolUse` hook runs after a tool and can record state.

**TODO 1 (about 6 minutes).** Complete `gate_offer`, the `PreToolUse` gate. If the tool is
`send_offer` and the candidate is **not** in `CLEARED`, return a deny decision; otherwise return `{}`.
The deny shape is provided in the comment.

In [ ]:
# ===== TODO 1 - the gate: deny an offer before the background check =====
async def gate_offer(input_data, tool_use_id, context):   # runs BEFORE a tool
    name = input_data["tool_name"].split("__")[-1]        # the bare tool name
    cid = input_data.get("tool_input", {}).get("candidate_id")   # which candidate

    # 👉 TODO 1: if name == "send_offer" and cid not in CLEARED, print a note and return the deny dict:
    #    return {"hookSpecificOutput": {
    #        "hookEventName": "PreToolUse",
    #        "permissionDecision": "deny",
    #        "permissionDecisionReason": "Run the background check first."}}

    return {}                                             # otherwise allow

**Self-check (offline).** Awaits the gate with a mock payload: an uncleared offer must be
denied; once cleared, it must be allowed.

In [ ]:
# ===== self-check for TODO 1 =====
CLEARED.clear()
payload = {"tool_name": "mcp__hr__send_offer", "tool_input": {"candidate_id": "C1", "salary": 150000}}
denied = run_async(lambda: gate_offer(payload, "tid", None))
assert denied.get("hookSpecificOutput", {}).get("permissionDecision") == "deny", "uncleared offer must be denied"
CLEARED.add("C1")
allowed = run_async(lambda: gate_offer(payload, "tid", None))
assert allowed == {}, "a cleared offer must be allowed"
CLEARED.clear()
print("TODO 1 checks passed")

**TODO 2 (about 4 minutes).** Complete `record_clearance`, the `PostToolUse` recorder. When a
`background_check` has just run for a candidate, add that candidate to `CLEARED`, which is what later
unlocks the offer. A recorder observes and never blocks, so it returns `{}`.

In [ ]:
# ===== TODO 2 - record a clearance so the gate can later allow the offer =====
async def record_clearance(input_data, tool_use_id, context):   # runs AFTER a tool
    name = input_data["tool_name"].split("__")[-1]              # the bare tool name
    cid = input_data.get("tool_input", {}).get("candidate_id")  # which candidate

    # 👉 TODO 2: if name == "background_check" and cid, add cid to CLEARED and print a note

    return {}                                                   # observe only, never blocks

**Self-check (offline).** Runs the recorder for C1, then confirms the gate now allows C1's
offer, the full prerequisite flow, decided entirely in code.

In [ ]:
# ===== self-check for TODO 2 =====
CLEARED.clear()
run_async(lambda: record_clearance(
    {"tool_name": "mcp__hr__background_check", "tool_input": {"candidate_id": "C1"}}, "tid", None))
assert "C1" in CLEARED, "the recorder must add C1 to CLEARED"
after = run_async(lambda: gate_offer(
    {"tool_name": "mcp__hr__send_offer", "tool_input": {"candidate_id": "C1", "salary": 150000}}, "tid", None))
assert after == {}, "after clearance, the gate should allow the offer"
CLEARED.clear()
print("TODO 2 checks passed")

---

### 🎯 Part B - clean the output, enforce the limit

**TODO 3 (about 5 minutes).** Complete `normalize()`, the pure-Python transform. Turn a Unix
timestamp under `last_activity_ts` into an ISO string under `last_activity_time`, and turn a numeric
`stage` into its word from `STAGE_NAMES`. Leave everything else untouched, and do not mutate the
input.

In [ ]:
# ===== TODO 3 - normalize a raw ATS payload: Unix -> ISO, code -> word =====
def normalize(raw):                                # raw tool payload -> a clean payload
    out = dict(raw)                                # copy so we never mutate the input
    # 👉 TODO 3a: if "last_activity_ts" in out, pop it and set
    #    out["last_activity_time"] = datetime.fromtimestamp(<ts>, timezone.utc).isoformat()
    # 👉 TODO 3b: if out.get("stage") in STAGE_NAMES, replace out["stage"] with the word
    return out

print("raw:  ", {"candidate_id": "C1", "last_activity_ts": 1718000000, "stage": 3})
print("clean:", normalize({"candidate_id": "C1", "last_activity_ts": 1718000000, "stage": 3}))

**Self-check (offline).**

In [ ]:
# ===== self-check for TODO 3 =====
n = normalize({"candidate_id": "C1", "last_activity_ts": 1718000000, "stage": 3})
assert "last_activity_ts" not in n and "last_activity_time" in n, "convert the timestamp"
assert n["stage"] == "interview", "stage 3 should become 'interview'"
assert n["candidate_id"] == "C1", "other fields must be untouched"
print("TODO 3 checks passed:", n)

**This cell:** the `PostToolUse` **normalization hook** (provided). After `get_candidate_record`
runs, it reads the raw output, calls your `normalize()`, and returns the clean view as
`additionalContext`. A `PostToolUse` hook cannot rewrite a result that already ran, so it **appends**
the cleaned companion the model then reasons over.

In [ ]:
# ===== PostToolUse: append a normalized view of the raw output (provided) =====
def _output_text(tool_output):                     # pull text out of whatever shape the output has
    if isinstance(tool_output, str):  return tool_output
    if isinstance(tool_output, dict): tool_output = tool_output.get("content", tool_output)
    if isinstance(tool_output, list):
        return "".join(b.get("text", "") for b in tool_output if isinstance(b, dict))
    return json.dumps(tool_output) if tool_output is not None else ""

async def normalize_output(input_data, tool_use_id, context):   # runs AFTER a tool
    if input_data["tool_name"].split("__")[-1] != "get_candidate_record":
        return {}                                               # only clean this tool's output
    try:
        raw = json.loads(_output_text(input_data.get("tool_output")))
    except Exception:
        return {}
    clean = normalize(raw)                                      # apply YOUR transform
    print("  [hook] appended normalized view:", clean)
    return {"hookSpecificOutput": {"hookEventName": "PostToolUse",
            "additionalContext": "normalized: " + json.dumps(clean)}}

# quick offline check of the hook
_out = run_async(lambda: normalize_output(
    {"tool_name": "mcp__hr__get_candidate_record",
     "tool_output": json.dumps({"candidate_id": "C1", "last_activity_ts": 1718000000, "stage": 3})}, "t", None))
assert "normalized" in _out["hookSpecificOutput"]["additionalContext"]
print("normalize hook OK")

**TODO 4 (about 4 minutes).** Complete `check_band()`, the compliance rule. An offer at or under
`BAND_LIMIT` is allowed; anything over needs a human. Return a `(allowed, reason)` tuple. The provided
`PreToolUse` gate below reuses this exact function, so the rule lives in one place.

In [ ]:
# ===== TODO 4 - the compliance rule: a salary-band threshold =====
def check_band(salary):                            # salary -> (allowed?, reason)
    # 👉 TODO 4: if salary > BAND_LIMIT, return (False, f"offer {salary} over {BAND_LIMIT} -> needs approval")
    #            otherwise return (True, "within band")
    return (True, "within band")                   # replace with the real check

for s in [120000, 180000, 220000]:
    print(f"{s:>7}:", check_band(s))

**Self-check (offline).**

In [ ]:
# ===== self-check for TODO 4 =====
assert check_band(150000)[0] is True, "150k is within the 180k band"
assert check_band(180000)[0] is True, "at the limit is allowed"
assert check_band(220000)[0] is False, "220k is over the band"
print("TODO 4 checks passed")

**This cell:** the `PreToolUse` **compliance gate** (provided). Before `send_offer` runs, it
checks the amount against the band with your `check_band()` and denies it if over. This intercepts an
invalid action before it can happen, which is the only place you can truly stop it.

In [ ]:
# ===== PreToolUse: block an offer over the compliance band (provided) =====
async def enforce_band(input_data, tool_use_id, context):   # runs BEFORE a tool
    if input_data["tool_name"].split("__")[-1] != "send_offer":
        return {}                                           # only guard offers
    salary = input_data.get("tool_input", {}).get("salary", 0)
    ok, reason = check_band(salary)                         # the same rule as offline
    if not ok:
        print("  [gate]", reason)
        return {"hookSpecificOutput": {"hookEventName": "PreToolUse",
                "permissionDecision": "deny", "permissionDecisionReason": reason}}
    return {}

# quick offline check of the gate
_over = run_async(lambda: enforce_band(
    {"tool_name": "mcp__hr__send_offer", "tool_input": {"candidate_id": "C1", "salary": 220000}}, "t", None))
_under = run_async(lambda: enforce_band(
    {"tool_name": "mcp__hr__send_offer", "tool_input": {"candidate_id": "C1", "salary": 150000}}, "t", None))
assert _over["hookSpecificOutput"]["permissionDecision"] == "deny" and _under == {}
print("compliance gate OK")

**This cell:** the **structured escalation** (provided). When a case needs a human, for example
an over-band offer, a consistent shape (candidate, root cause, recommended actions, priority) makes the
hand-off reliable. Building the packet in code, not free prose, is what keeps every hand-off
consistent.

In [ ]:
# ===== structured escalation: a fixed contract, built and validated (provided) =====
ESCALATION_SCHEMA = {
    "type": "object",
    "properties": {
        "candidate": {"type": "object",
                      "properties": {"name": {"type": "string"}, "candidate_id": {"type": "string"}},
                      "required": ["name", "candidate_id"]},
        "root_cause": {"type": "string"},
        "recommended_actions": {"type": "array", "items": {"type": "string"}},
        "priority": {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": ["candidate", "root_cause", "recommended_actions", "priority"],
}

def build_escalation(candidate_id, root_cause, actions, priority):
    return {
        "candidate": {"name": CANDIDATES.get(candidate_id, {}).get("name", "unknown"),
                      "candidate_id": candidate_id},
        "root_cause": root_cause, "recommended_actions": actions, "priority": priority,
    }

def validate_escalation(pkt):
    return (all(k in pkt for k in ESCALATION_SCHEMA["required"]) and
            all(k in pkt["candidate"] for k in ["name", "candidate_id"]))

esc = build_escalation("C1", "Requested salary above the band; needs sign-off.",
                       ["Route to hiring manager", "Approve or counter within 24h"], "high")
print(json.dumps(esc, indent=2))
print("valid:", validate_escalation(esc))

**This cell:** the **live Agent SDK wiring** (provided). It defines the tools, bundles them, and
wires all the hooks into one options object with `HookMatcher`. Live, this is the enforced workflow;
offline it is skipped. The hook logic you wrote above is exactly what runs here.

In [ ]:
# ===== the SDK tools + hooked options (provided, live) =====
try:
    from claude_agent_sdk import (query, ClaudeAgentOptions, tool, create_sdk_mcp_server, HookMatcher,
                                  AssistantMessage, ResultMessage, TextBlock, ToolUseBlock)
    SDK_OK = True

    @tool("background_check", "Run the background check for a candidate.", {"candidate_id": str})
    async def sdk_bg(args):
        return {"content": [{"type": "text", "text": json.dumps(CANDIDATES.get(args["candidate_id"], {}))}]}

    @tool("get_candidate_record", "Get the latest ATS record (raw fields).", {"candidate_id": str})
    async def sdk_record(args):
        raw = {"candidate_id": args["candidate_id"], "last_activity_ts": 1718000000, "stage": 3}
        return {"content": [{"type": "text", "text": json.dumps(raw)}]}

    @tool("send_offer", "Send an offer to a candidate.", {"candidate_id": str, "salary": float})
    async def sdk_offer(args):
        return {"content": [{"type": "text", "text": f"offer sent to {args['candidate_id']} at {args['salary']}"}]}

    hr = create_sdk_mcp_server(name="hr", version="1.0.0", tools=[sdk_bg, sdk_record, sdk_offer])
    GUARDED = ClaudeAgentOptions(
        model=MODEL, mcp_servers={"hr": hr},
        allowed_tools=["mcp__hr__background_check", "mcp__hr__get_candidate_record", "mcp__hr__send_offer"],
        hooks={"PreToolUse":  [HookMatcher(hooks=[gate_offer, enforce_band])],   # prerequisite + compliance
               "PostToolUse": [HookMatcher(hooks=[record_clearance, normalize_output])]})  # record + normalize

    async def stream_run(options, prompt):
        print("USER:", prompt); answer = ""
        async for message in query(prompt=prompt, options=options):
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, ToolUseBlock): print("  -> tool:", block.name.split("__")[-1], block.input)
                    elif isinstance(block, TextBlock):  answer = block.text
        print("ANSWER:", answer); return answer
    print("guarded workflow ready")
except Exception as e:
    SDK_OK = False
    print("Agent SDK not available offline; the hooks above were already tested with mock payloads.")

In [ ]:
# ===== run the guarded workflow live (cold offer denied; cleared + within-band allowed) =====
if RUN_LIVE and SDK_OK:
    CLEARED.clear()
    print("--- cold offer (no background check) ---")
    run_async(lambda: stream_run(GUARDED, "Send a $150000 offer to candidate C1."))       # gate denies
    print("--- background check, then a within-band offer ---")
    run_async(lambda: stream_run(GUARDED, "Run the background check for C1, then send a $150000 offer."))
    print("--- an over-band offer ---")
    run_async(lambda: stream_run(GUARDED, "Run the background check for C1, then send a $220000 offer."))  # compliance denies
else:
    print("[offline] expected live:")
    print("  cold offer  -> gate_offer DENIES (C1 not cleared)")
    print("  after check -> record_clearance unlocks; a $150000 offer is ALLOWED")
    print("  $220000     -> enforce_band DENIES as over the band; escalate for sign-off")

---

### Anti-patterns to avoid

| anti-pattern | what to do instead |
|---|---|
| trust a prompt to enforce an offer rule | gate the tool in code with a `PreToolUse` deny |
| track prerequisites in the prompt | record them in code (a `PostToolUse` hook + a set) |
| let the model reason over raw Unix time and codes | normalize at the hook layer into ISO and words |
| try to "undo" a bad action in `PostToolUse` | block it in `PreToolUse`; PostToolUse cannot undo |
| put a salary limit only in the prompt | enforce the threshold in a `PreToolUse` deny |
| hardcode the same rule in two places | share one `check_band` between offline and the hook |
| hand off cases as free prose | fill a structured schema so every hand-off is consistent |

**Lesson:** the prompt asks; your **code decides**. A `PreToolUse` gate turns "clear before you
offer" and "stay within band" from suggestions into guarantees, a `PostToolUse` hook records the
prerequisite and appends a normalized view, and a structured escalation schema keeps every hand-off
consistent. Prevention lives in `PreToolUse`; `PostToolUse` shapes and records but cannot undo.

---

## Recap - guarantees in code

| Piece | In this lab | Course topic |
|---|---|---|
| PreToolUse gate | deny `send_offer` until cleared | programmatic prerequisite enforcement (Lab 1) |
| PostToolUse recorder | mark a candidate cleared after `background_check` | control points in the lifecycle (Lab 1) |
| Enforcement contrast | ungated runs vs gated blocks | prompt-based vs programmatic reliability (Lab 1) |
| PostToolUse normalizer | append a clean view (Unix to ISO, code to word) | transform output before the model (Lab 2) |
| PreToolUse compliance gate | deny an over-band offer | intercept and block invalid actions (Lab 2) |
| Escalation schema | candidate, root cause, actions, priority | structured handoff patterns (Lab 1) |

**Try it next:** word the cold offer as an emergency and confirm the gate still denies it. Then lower
`BAND_LIMIT` to 100000 and watch the $150000 offer get blocked too, then escalate it with
`build_escalation`.